# Assignment 5: Neural Networks

---

## Task 2) RNN for Classification

The theses dataset also contains types (diploma, bachelor, master) and categories (internal/external) for each thesis. 
In this part, we want to classify whether the thesis is bachelor or master; and if it's internal or external. 
Since PyTorch provides most things sort-of out of the box, we want you to compare the following Recurrent Neural Network variation: 
[RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html), [GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html), [LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html), and Bidirectional-[LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html) by using the `bidirectional` flag.
The basic setup as well as some code and steps can be reused from your solution for the language modeling task.

### Data

Download the `theses.csv` data set from the `Supplemental Materials` in the `Files` section of our Microsoft Teams group.
This dataset consists of approx. 3,000 theses topics chosen by students in the past.
Here are some examples of the file content:

```
27.10.94;14.07.95;1995;intern;Diplom;DE;Monte Carlo-Simulation für ein gekoppeltes Round-Robin-System;
04.11.94;14.03.95;1995;intern;Diplom;DE;Implementierung eines Testüberdeckungsgrad-Analysators für RAS;
01.11.20;01.04.21;2021;intern;Bachelor;DE;Landessprachenerkennung mittels X-Vektoren und Meta-Klassifikation;
```

### Basic Setup

For the assignment on Recurrent Neural Networks, we'll (again) heavily use [PyTorch](https://pytorch.org) as go-to Deep Learning library.
Here, we'll rely on the RNN and Embedding modules already implemented by PyTorch.
You can imagine the Embedding layer as a simple lookup table that stores embeddings of a fixed dictionary and size (quite similar to the Word2Vec parameters we've trained in assignment 2).
Head over to the [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) and [Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) modules to gain some understanding of their functionality.
Code for processing data samples, batching, converting to tensors, etc. can get messy and hard to maintain. 
Therefore, you can use PyTorch's [Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html). 
Get familiar with the basics of data handling, as it will help you for upcoming assignments.
As always, you can use [NumPy](https://numpy.org) and [Pandas](https://pandas.pydata.org) for data handling etc.

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [47]:
# Dependencies
import os
import tqdm
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

### Prepare the Data

1.1 Spend some time on preparing the dataset. It may be helpful to lower-case the data and to filter for German titles. The format of the CSV-file should be:

```
Anmeldedatum;Abgabedatum;JahrAkademisch;Art;Grad;Sprache;Titel;Abstract
```

1.2 Create the vocabulary from the prepared dataset. You'll need it for the modeling part such as nn.Embedding.

1.3 Filter out all diploma theses; they might be too easy to spot because they only cover "old" topics.

1.4 Create a PyTorch Dataset class which handles your tokenized data with respect to input and (class) labels.

In [32]:
def load_theses_dataset(filepath):
    """Loads all theses instances and returns them as a dataframe."""
    ### YOUR CODE HERE
    
    return pd.read_csv(filepath, sep=";")
    
    ### END YOUR CODE

In [33]:
### Notice: Think about start and end of sentence tokens

def preprocess(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Preprocesses and tokenizes the given theses titles for further use."""
    ### YOUR CODE HERE
    pattern = r"\b[\wäöüÄÖÜß]+(?:[-'][\wäöüÄÖÜß]+)*\b"
    labels = {
        "intern_bachelor": 2,
        "extern_bachelor": 3,
        "intern_master": 0,
        "extern_master": 1,
        "intern_diplom": 4,
        "extern_diplom": 5,
    }

    df_german = dataframe[dataframe.Sprache == "DE"].copy()

    tokens = [re.findall(pattern, str(row).lower(), flags=re.UNICODE) 
              for row in df_german.Titel]

    df_german["tokens"] = tokens
    df_german["labels"] = (df_german["Art"].str.lower() + "_" + df_german["Grad"].str.lower()).map(labels)

    return df_german

    ### END YOUR CODE

In [34]:
df = load_theses_dataset("C:\\Users\\Felix\\PythonProjects\\seqlrn_assignments\\5-nnet_rnn\\data\\theses2022.csv")
df = preprocess(df)
df_no_diplom = df[df.Grad != "Diplom"].copy()

# vocabulary for whole dataframe so there are no embedding errors later
PAD_TOKEN = "<PAD>"
vocabulary = set([PAD_TOKEN])
for title in df.tokens:
    vocabulary.update(title)
vocab_size = len(vocabulary)

word2idx = {
    word: idx for idx, word in enumerate(vocabulary)
}

idx2word = {
    idx: word for word, idx in word2idx.items()
}

In [35]:
### TODO: 1.3 Implement the PyTorch theses dataset
### Notice: It is possible to solve the task without this class.
### Notice: However, with respect to DataLoaders it makes your life easier.

### YOUR CODE HERE

class ThesesClassificationDataset(Dataset):
    def __init__(self, data, labels, word2idx):
        self.data = data.apply(lambda title: [word2idx[token] for token in title]).to_list()
        self.labels = labels.to_list()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (torch.tensor(self.data[idx]), 
                torch.tensor(self.labels[idx]))
    
### END YOUR CODE

### Train and Evaluate

2.1 Implement the RNN for Classification. Therefore, you can use the nn.Module and overwrite the forward function.

2.2 Train and evaluate your models with 5-fold cross-validation. As in RNN-LM, you can either learn the embeddings from scratch or reuse the ones from word2vec.

2.3 Assemble a table: Recall/Precision/F1 measure for each of the mentioned RNN variants (RNN, GRU, LSTM). Which one works best?

2.4 Bonus: Apply your best classifier to the remaining diploma theses; are those on average more bachelor or master? :-)

In [36]:
### TODO: 2.1 Implement the RNN classifier (nn.Module)
### Notice: Think about padding for batch sizes > 1
### Notice: 'torch.nn.utils.rnn' provides functionality

### YOUR CODE HERE

class RNN_Classifier(nn.Module):
    def __init__(self, word2idx, embedding_dim, hidden_dim, num_layers, num_classes, recurrence_type="rnn", bidirectional=False):
        super(RNN_Classifier, self).__init__()

        self.embedding = nn.Embedding(len(word2idx), embedding_dim, padding_idx=word2idx["<PAD>"])
        self.recurrence_type = recurrence_type.lower()
        self.bidirectional = bidirectional
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        match recurrence_type:
            case "rnn":
                self.rnn = nn.RNN(
                    input_size=embedding_dim,
                    hidden_size=hidden_dim,
                    num_layers=num_layers,
                    batch_first=True,
                    bidirectional=bidirectional
                )            
            case "gru":
                self.rnn = nn.GRU(
                    input_size=embedding_dim,
                    hidden_size=hidden_dim,
                    num_layers=num_layers,
                    batch_first=True,
                    bidirectional=bidirectional
                )  
            case "lstm":
                self.rnn = nn.LSTM(
                    input_size=embedding_dim,
                    hidden_size=hidden_dim,
                    num_layers=num_layers,
                    batch_first=True,
                    bidirectional=bidirectional
                )  
            case _:
                raise ValueError(f"Not a valid recurrence_type: {recurrence_type}")

        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)
    
    def forward(self, X, hidden=None):
        embeddings = self.embedding(X) 
        _, hidden = self.rnn(embeddings, hidden)

        if self.recurrence_type == "lstm":
            hidden = hidden[0]  # only use h_n
        if self.bidirectional:
            last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            last_hidden = hidden[-1]  

        logits = self.fc(last_hidden)
        return logits, hidden

### END YOUR CODE

In [37]:
### TODO: 2.2 Implement the train functionality
### Notice: If you want, you can also combine train and eval functionality

def train(model: RNN_Classifier, dataloader: DataLoader, optimizer: optim.Optimizer, criterion, device):
    """Trains the RNN-Classifier for one epoch."""
    ### YOUR CODE HERE
    epoch_loss = 0
    model.to(device)
    model.train()

    for sample, target in dataloader:
        optimizer.zero_grad()

        sample = sample.to(device)
        target = target.to(device)
        pred, _ = model(sample)
        pred = pred.squeeze(0)
        loss = criterion(pred, target.squeeze())
        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss
    ### END YOUR CODE

In [38]:
### TODO: 2.2 Implement the evaluation functionality
### Notice: If you want, you can also combine train and eval

def eval(model: RNN_Classifier, dataloader: DataLoader, criterion, device):
    """Evaluates the optimized RNN-Classifier."""
    eval_loss = 0
    all_preds = []
    model.to(device)
    model.eval()

    with torch.no_grad():
        for sample, target in dataloader:
            sample = sample.to(device)
            target = target.to(device)
            
            pred, _ = model(sample)
            pred = pred.squeeze(0)

            loss = criterion(pred, target.squeeze())

            eval_loss += loss.item()
            
            all_preds.append(torch.argmax(torch.softmax(pred, dim=0), dim=0).tolist())

    return eval_loss, all_preds
    ### END YOUR CODE

In [39]:
### TODO: 2.2 Initialize and train the RNN-Classifier for X epochs

# For split reproducibility
# Use 5-fold cross validation
SEED = 666

EPOCHS = 10

print(torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # 'cpu', 'mps' or 'cuda'

LABEL_COL = "Grad"

LEARN_RATE = 0.0001

### YOUR CODE HERE
data = df_no_diplom["tokens"]
labels = df_no_diplom["labels"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
skf.get_n_splits(data, labels)

fold_models = {
}

fold_losses = {
}

for i, (train_index, test_index) in enumerate(skf.split(data, labels)):
    print(f"Fold {i}:")
    # Your language model
    rnn = RNN_Classifier(word2idx, embedding_dim=300, hidden_dim=64, num_layers=1, num_classes=4)
    gru = RNN_Classifier(word2idx, embedding_dim=300, hidden_dim=64, num_layers=1, num_classes=4, recurrence_type="gru")
    lstm = RNN_Classifier(word2idx, embedding_dim=300, hidden_dim=64, num_layers=1, num_classes=4, recurrence_type="lstm")
    bi_lstm = RNN_Classifier(word2idx, embedding_dim=300, hidden_dim=64, num_layers=1, num_classes=4, recurrence_type="lstm", bidirectional=True)

    # Your loss function
    criterion = nn.CrossEntropyLoss()

    # Your optimizer (optim.SGD should be okay)
    rnn_optimizer = optim.Adam(rnn.parameters(), lr=LEARN_RATE)
    gru_optimizer = optim.Adam(gru.parameters(), lr=LEARN_RATE)
    lstm_optimizer = optim.Adam(lstm.parameters(), lr=LEARN_RATE)
    bi_lstm_optimizer = optim.Adam(bi_lstm.parameters(), lr=LEARN_RATE)

    # Use batch_size=1 if you want to avoid padding handling
    train_dataset = ThesesClassificationDataset(data.iloc[train_index], labels.iloc[train_index], word2idx)
    train_dataloader = DataLoader(train_dataset)

    # Use batch_size=1 if you want to avoid padding handling
    test_dataset = ThesesClassificationDataset(data.iloc[test_index], labels.iloc[test_index], word2idx)
    test_dataloader = DataLoader(test_dataset)

    for e in range(1, EPOCHS+1):
        # i dont know if this is the right way to do.......
        train_loss_rnn = train(rnn, train_dataloader, rnn_optimizer, criterion, DEVICE)
        train_loss_gru = train(gru, train_dataloader, gru_optimizer, criterion, DEVICE)
        train_loss_lstm = train(lstm, train_dataloader, lstm_optimizer, criterion, DEVICE)
        train_loss_bi_lstm = train(bi_lstm, train_dataloader, bi_lstm_optimizer, criterion, DEVICE)

        print(f"    Epoch {e} RNN loss: {train_loss_rnn}")
        print(f"    Epoch {e} GRU loss: {train_loss_gru}")
        print(f"    Epoch {e} LSTM loss: {train_loss_lstm}")
        print(f"    Epoch {e} LSTM bidirectional loss: {train_loss_bi_lstm}")


    rnn_eval_loss, rnn_preds = eval(rnn, test_dataloader, criterion, DEVICE)
    gru_eval_loss, gru_preds = eval(gru, test_dataloader, criterion, DEVICE)
    lstm_eval_loss, lstm_preds = eval(lstm, test_dataloader, criterion, DEVICE)
    bi_lstm_eval_loss, bi_lstm_preds = eval(bi_lstm, test_dataloader, criterion, DEVICE)

    print(f"    Fold {i} RNN eval loss: {rnn_eval_loss}")
    print(f"    Fold {i} GRU eval loss: {gru_eval_loss}")
    print(f"    Fold {i} LSTM eval loss: {lstm_eval_loss}")
    print(f"    Fold {i} LSTM bidirectional eval loss: {bi_lstm_eval_loss}")

    fold_losses[i] = {
        "rnn": rnn_eval_loss,
        "gru": gru_eval_loss,
        "lstm": lstm_eval_loss,
        "bi_lstm": bi_lstm_eval_loss,
    }

    fold_models[i] = {
        "rnn": rnn,
        "gru": gru,
        "lstm": lstm,
        "bi_lstm": bi_lstm,       
    }


### END YOUR CODE

False
Fold 0:
    Epoch 1 RNN loss: 2255.2049897909164
    Epoch 1 GRU loss: 2112.534750685096
    Epoch 1 LSTM loss: 2112.087718516588
    Epoch 1 LSTM bidirectional loss: 2039.2577731832862
    Epoch 2 RNN loss: 1905.8307138085365
    Epoch 2 GRU loss: 1767.1943700537086
    Epoch 2 LSTM loss: 1802.3825417757034
    Epoch 2 LSTM bidirectional loss: 1756.0201688967645
    Epoch 3 RNN loss: 1706.8904707729816
    Epoch 3 GRU loss: 1592.5387379005551
    Epoch 3 LSTM loss: 1633.6226412653923
    Epoch 3 LSTM bidirectional loss: 1567.9398157522082
    Epoch 4 RNN loss: 1562.2987318262458
    Epoch 4 GRU loss: 1410.4559959247708
    Epoch 4 LSTM loss: 1446.5536906197667
    Epoch 4 LSTM bidirectional loss: 1344.548140944913
    Epoch 5 RNN loss: 1421.372437339276
    Epoch 5 GRU loss: 1208.5634376592934
    Epoch 5 LSTM loss: 1239.5627281144261
    Epoch 5 LSTM bidirectional loss: 1089.860655629076
    Epoch 6 RNN loss: 1277.035652583465
    Epoch 6 GRU loss: 993.5528301075101
    Epoch 6

In [ ]:
### TODO: 2.3 Compare the results of various RNN variants (classification metrics)

### YOUR CODE HERE

def plot_metrics(model_name, preds, targets, multi_class=True):
    average = "macro" if multi_class else None

    f1 = f1_score(targets, preds, average=average)
    prec = precision_score(targets, preds, average=average )
    recall = recall_score(targets, preds, average=average)

    print(f"Model    | {model_name}  ")
    print(f"---------|----------------------")
    print(f"F1 Score | {f1:.2f} ")
    print(f"Precision| {prec:.2f}")
    print(f"Recall   | {recall:.2f}")
    print(f"---------|----------------------")


def get_best_models_per_type(fold_losses, fold_models):
    best_models = {}
    best_losses = {}

    for fold_idx, losses in fold_losses.items():
        for model_type, loss in losses.items():
            if model_type not in best_losses or loss < best_losses[model_type]:
                best_losses[model_type] = loss
                best_models[model_type] = fold_models[fold_idx][model_type]

    for model_type in best_models:
        print(f"Best {model_type} model with eval loss: {best_losses[model_type]:.4f}")

    return best_models

best_models = get_best_models_per_type(fold_losses, fold_models)

X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.33, random_state=666)

test_dataset = ThesesClassificationDataset(X_test, y_test, word2idx)
test_dataloader = DataLoader(test_dataset)

_, rnn_preds = eval(best_models["rnn"], test_dataloader, criterion, DEVICE)
_, gru_preds = eval(best_models["gru"], test_dataloader, criterion, DEVICE)
_, lstm_preds = eval(best_models["lstm"], test_dataloader, criterion, DEVICE)
_, bi_lstm_preds = eval(best_models["bi_lstm"], test_dataloader, criterion, DEVICE)

print(rnn_preds)
print(y_test)
plot_metrics("RNN", preds=rnn_preds, targets=y_test)
plot_metrics("GRU", preds=gru_preds, targets=y_test)
plot_metrics("LSTM", preds=lstm_preds, targets=y_test)
plot_metrics("Bidirectional LSTM", preds=bi_lstm_preds, targets=y_test)

### END YOUR CODE

[2, 0, 3, 3, 2, 2, 2, 3, 2, 3, 2, 3, 3, 3, 3, 3, 3, 1, 2, 3, 3, 3, 3, 3, 1, 3, 1, 2, 1, 3, 2, 3, 2, 3, 3, 3, 0, 2, 1, 3, 2, 2, 3, 0, 3, 3, 3, 3, 3, 2, 0, 2, 3, 3, 3, 2, 3, 1, 3, 1, 3, 2, 1, 3, 1, 3, 2, 1, 3, 1, 2, 0, 2, 3, 2, 3, 2, 2, 0, 0, 3, 2, 3, 2, 2, 1, 1, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 1, 3, 2, 3, 3, 2, 3, 2, 1, 3, 0, 2, 2, 3, 1, 3, 1, 1, 3, 3, 3, 2, 2, 3, 3, 3, 1, 2, 2, 1, 3, 3, 3, 0, 1, 2, 1, 2, 2, 2, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 1, 2, 1, 3, 2, 2, 1, 3, 3, 3, 2, 0, 1, 3, 2, 2, 2, 1, 3, 2, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 2, 3, 2, 1, 3, 2, 1, 3, 2, 3, 1, 2, 3, 3, 0, 2, 3, 3, 3, 1, 1, 2, 2, 2, 1, 2, 3, 3, 3, 3, 3, 2, 3, 2, 3, 0, 3, 2, 1, 3, 2, 2, 3, 0, 2, 3, 3, 0, 3, 2, 3, 0, 2, 3, 3, 3, 2, 2, 3, 3, 2, 2, 3, 3, 1, 3, 3, 2, 2, 3, 1, 3, 3, 2, 3, 3, 2, 3, 2, 3, 3, 2, 1, 3, 3, 3, 3, 1, 3, 3, 3, 3, 2, 3, 1, 3, 2, 3, 3, 2, 2, 2, 3, 3, 3, 3, 2, 2, 3, 0, 2, 3, 2, 3, 1, 3, 3, 1, 1, 2, 3, 3, 2, 3, 3, 3, 2, 3, 2, 1, 3, 3, 1, 3, 1, 2, 3, 1, 2, 3, 3, 3, 1, 3, 3, 3, 1, 3, 

In [ ]:
### TODO: 2.4 (Optional) Apply your best classifier to the diploma theses

### YOUR CODE HERE



### END YOUR CODE